In [1]:
import pandas as pd

import sys
import os

# sobe um nível (se o notebook estiver em src)
sys.path.append(os.path.abspath(".."))
from config.constants import NAICS, COUNTRY_NAMES

_FINANCIAL_COLS = [
    "incident_id", "total_loss_usd", "direct_loss_usd", "recovery_cost_usd",
    "legal_fees_usd", "regulatory_fine_usd", "insurance_payout_usd",
    "ransom_demanded_usd", "ransom_paid_usd", "inflation_adjusted_usd",
]

_MARKET_COLS = [
    "incident_id", "abnormal_return_1d", "abnormal_return_7d", "abnormal_return_30d",
    "car_0_to_30", "car_0_to_90", "days_to_price_recovery", "market_cap_at_disclosure",
    "volume_ratio_disclosure", "pre_incident_volatility_30d", "post_incident_volatility_30d",
    "price_7d_before", "price_disclosure_day", "price_30d_after", "sector_index",
]

In [2]:
# %%
from pathlib import Path

# Define raiz do projeto (sobe um nível a partir de src)
ROOT = Path().resolve().parent
DATA_PATH = ROOT / "data"

fi = pd.read_csv(DATA_PATH / "financial_impact.csv")
im = pd.read_csv(DATA_PATH / "incidents_master.csv")
mi = pd.read_csv(DATA_PATH / "market_impact.csv")

# ── Transformações em incidents_master ───────────────────────
im["incident_date"] = pd.to_datetime(im["incident_date"], errors="coerce")
im["year"] = im["incident_date"].dt.year

im["industry_name"] = (
    im["industry_primary"].astype(str).map(NAICS)
    .fillna(im["industry_primary"].astype(str))
)

im["country_name"] = (
    im["country_hq"].map(COUNTRY_NAMES)
    .fillna(im["country_hq"])
)

im["attack_vector_primary"] = (
    im["attack_vector_primary"].str.replace("_", " ").str.title()
)

# ── Merges ────────────────────────────────────────────────────
df = im.merge(fi[_FINANCIAL_COLS], on="incident_id", how="left")
df = df.merge(mi[_MARKET_COLS], on="incident_id", how="left")

In [4]:
df["country_name"].value_counts()

country_name
United States     298
United Kingdom     90
Germany            50
Canada             48
France             40
Australia          37
Japan              35
India              25
Italy              20
Brazil             17
Mexico             16
Israel             16
Switzerland        15
Singapore          14
South Korea        13
AT                 12
NL                 12
DK                  9
BE                  9
NO                  9
IE                  8
TW                  5
ZA                  5
HK                  5
SE                  5
AE                  5
NZ                  4
ES                  4
MY                  4
PT                  4
PH                  3
CZ                  3
CO                  2
FI                  2
PL                  2
TH                  2
CL                  1
AR                  1
Name: count, dtype: int64